# **Assignment 2: Secure ML Pipeline**

**Course:** Introduction to Data Security Pr.  
**Total Points:**  10  
**Estimated Time:** 90 min

---

**Submission Instructions**

1. **Make sure your notebook is complete** (Run all cells before submitting).

2. **Save your final notebook** (Use the filename format: **`Assignment_2_FirstName_LastName_NeptunCode.ipynb`**).

3. **Upload your notebook to Microsoft Teams**
   - Go to the **Teams channel**.
   - Open the folder named **`Assignment_2`**.
   - Upload your `.ipynb` file into **`Submissions`** folder.

4. **Verify your upload**
   - Make sure the file appears in the folder.
   - Confirm that the correct version was uploaded.

**Assignment Goals:**

1. **Design** a multi-threat secure ML system
2. **Implement** layered defenses: data pipeline, adversarial training, DP-SGD, input validation
3. **Evaluate** 6 attack types: evasion (FGSM, PGD), poisoning (label-flip, backdoor), trojans, inversion, MIA, sponge
4. **Defend** with detection mechanisms: trojan activation clustering, synthetic data, rate limiting
5. **Operationalize** security: monitoring, logging, incident response, deployment checklist

You are the ML security lead for a healthcare imaging startup. The company deploys a
cloud-hosted classifier for medical triage. Your system must withstand:

- **Evasion attacks:** Adversarial perturbations to fool predictions
- **Data poisoning:** Malicious training samples injected into the pipeline
- **Model trojans:** Backdoors injected into model weights via supply chain attacks
- **Membership inference:** Adversaries trying to detect patient data inclusion
- **Model inversion:** Privacy attacks to reconstruct training data from model outputs
- **Sponge attacks:** Resource exhaustion via crafted inputs

Your job: build a secure pipeline, quantify its performance, and operationalize defenses.

## Threat Model <a id="threat-model"></a>

| Layer | Threat | Goal | Attacker | Defense |
|-------|--------|------|----------|---------|
| **Data** | Label-flip poisoning | Degrade accuracy | Insider | Outlier detection |
| **Data** | Backdoor samples | Plant trojan trigger | Supply chain | Data sanitization |
| **Training** | Membership inference | Privacy leakage | Black-box | DP-SGD |
| **Model** | Weight trojans | Hidden functionality | Post-hoc compromise | Activation clustering |
| **Inference** | Evasion (FGSM/PGD) | Misclassification | White-box | Adversarial training |
| **Inference** | Model inversion | Reconstruct training data | Black-box | Differential privacy |
| **Inference** | Sponge attacks | DoS/resource exhaustion | Black-box | Input validation + rate limiting |
| **Ops** | Model theft | IP exposure | Reconnaissance | Logging + anomaly detection |

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision.transforms as transforms
from torchvision.datasets import MNIST

import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
from dataclasses import dataclass

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Keep the experiment lightweight so it is practical on a weak machine.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# MNIST is used here because it is small, standard, and easy to explain.
train_dataset = MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = MNIST(root='./data', train=False, download=True, transform=transform)

# Convert NumPy arrays to plain Python lists for cleaner Dataset handling.
train_indices = np.random.choice(len(train_dataset), 3000, replace=False).tolist()
test_indices = np.random.choice(len(test_dataset), 800, replace=False).tolist()

train_data = Subset(train_dataset, train_indices)
test_data = Subset(test_dataset, test_indices)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

print(f"Train: {len(train_data)}, Test: {len(test_data)}")

In [ ]:
# Simple CNN for MNIST classification.
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Wrap Python lists so they behave like standard PyTorch datasets.
class ListDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]

def dataset_labels(dataset):
    """Collect labels from any indexable dataset or subset."""
    return [dataset[i][1] for i in range(len(dataset))]

def binary_roc_auc(y_true, y_score):
    """Small dependency-free ROC-AUC implementation for binary scores."""
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    pos_scores = y_score[y_true == 1]
    neg_scores = y_score[y_true == 0]
    if len(pos_scores) == 0 or len(neg_scores) == 0:
        return 0.5
    comparisons = (pos_scores[:, None] > neg_scores[None, :]).mean()
    ties = (pos_scores[:, None] == neg_scores[None, :]).mean()
    return float(comparisons + 0.5 * ties)

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)
    return 100.0 * correct / total

## Baseline Model

Training a baseline model (without defenses) on clean data. This will be your reference before adding defenses.

In [ ]:
# Standard supervised training for the clean baseline.
def train_standard(model, loader, epochs=2):
    """Train model with standard SGD on clean or poisoned data."""
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        sample_count = 0

        for data, target in loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            batch_size = target.size(0)
            running_loss += loss.item() * batch_size
            sample_count += batch_size

        avg_loss = running_loss / max(sample_count, 1)
        print(f'Epoch {epoch + 1}/{epochs} - loss: {avg_loss:.4f}')

    return model

baseline_model = CNN().to(device)
baseline_loader = DataLoader(train_data, batch_size=64, shuffle=True)
train_standard(baseline_model, baseline_loader, epochs=2)

baseline_train_acc = evaluate(baseline_model, train_loader)
baseline_test_acc = evaluate(baseline_model, test_loader)
print(f'\nBaseline Model Accuracy: Train={baseline_train_acc:.2f}%, Test={baseline_test_acc:.2f}%')

## Secure Data Pipeline: Poisoning + Detection

- Simulate a label‑flip poisoning attack (use a small poison rate, e.g. 10%).
- Train a model on the poisoned training split and compare test accuracy with the baseline model.
- Compute class label distributions before and after poisoning and decide on a simple sanitization step (e.g., downsample overrepresented target label).

Expected output: poison count, label distribution summary, baseline vs poisoned test accuracy, and a one‑line sanitization decision.

In [ ]:
def poison_labels(dataset, poison_rate=0.1, target_label=0):
    """Flip a fraction of labels to one target label."""
    indices = set(np.random.choice(len(dataset), int(poison_rate * len(dataset)), replace=False).tolist())
    poisoned = []
    for i in range(len(dataset)):
        x, y = dataset[i]
        if i in indices:
            poisoned.append((x, target_label))
        else:
            poisoned.append((x, y))
    return poisoned, indices

poison_rate = 0.10
target_label = 0

# Step 1: create the poisoned training set.
poisoned_data, poisoned_indices = poison_labels(train_data, poison_rate=poison_rate, target_label=target_label)
poisoned_dataset = ListDataset(poisoned_data)
print(f'Poisoned samples (label flips): {len(poisoned_indices)} / {len(poisoned_dataset)}')

# Step 2: compare label distributions before and after poisoning.
orig_labels = dataset_labels(train_data)
poisoned_labels = dataset_labels(poisoned_dataset)
orig_dist = Counter(orig_labels)
poisoned_dist = Counter(poisoned_labels)
print('\nOriginal label counts:')
print(dict(orig_dist))
print('\nPoisoned label counts:')
print(dict(poisoned_dist))

# Step 3: train a quick model on poisoned data and compare test accuracy.
poisoned_loader = DataLoader(poisoned_dataset, batch_size=64, shuffle=True)
poisoned_model = CNN().to(device)
train_standard(poisoned_model, poisoned_loader, epochs=1)
poisoned_test_acc = evaluate(poisoned_model, test_loader)
print(f'\nBaseline Test Accuracy: {baseline_test_acc:.2f}%')
print(f'Poisoned-model Test Accuracy: {poisoned_test_acc:.2f}%')
print(f'Accuracy drop (poisoned - baseline): {poisoned_test_acc - baseline_test_acc:+.2f}pp')

# Step 4: simple sanitization rule based on target-label over-representation.
n = len(poisoned_dataset)
expected_prop = 1.0 / 10.0
prop_target = poisoned_dist.get(target_label, 0) / n
print(f'\nTarget label proportion after poisoning: {prop_target:.3f} (expected ~{expected_prop:.3f})')

if prop_target > expected_prop + 0.05:
    target_indices = [i for i, (_, y) in enumerate(poisoned_data) if y == target_label]
    keep_target = int(expected_prop * n)
    keep_set = set(target_indices[:max(0, keep_target)])
    sanitized_data = [sample for i, sample in enumerate(poisoned_data) if not (i in target_indices and i not in keep_set)]
    decision = f'DOWN_SAMPLE_TARGET - reduced target label to approx {expected_prop:.2f}'
else:
    sanitized_data = poisoned_data
    decision = 'KEEP_ALL - no large target-label imbalance detected'

sanitized_dataset = ListDataset(sanitized_data)
print('\nSanitization decision:', decision)
print(f'Sanitized dataset size: {len(sanitized_dataset)}')

## Robust Training: Adversarial + DP-SGD

- Implement robust training with FGSM-based adversarial examples.
- Add DP-SGD style clipping and noise in the training step.
- Compare secure model accuracy against baseline.

Expected output: secure train/test accuracy with clear defense configuration.

In [ ]:
@dataclass
class DefenseConfig:
    epsilon_adv: float = 0.1
    clip_norm: float = 1.0
    noise_multiplier: float = 0.35

def fgsm_attack(model, data, target, epsilon=0.1):
    """Single-step adversarial attack used during evaluation and training."""
    model.eval()
    perturbed_input = data.detach().clone().requires_grad_(True)
    output = model(perturbed_input)
    loss = nn.CrossEntropyLoss()(output, target)
    gradient = torch.autograd.grad(loss, perturbed_input)[0]
    perturbed = perturbed_input + epsilon * gradient.sign()
    return torch.clamp(perturbed, -3, 3).detach()

def dp_sgd_step(model, data, target, config: DefenseConfig):
    """Lightweight DP-style SGD: clip batch gradients, then add Gaussian noise."""
    model.train()
    criterion = nn.CrossEntropyLoss()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.clip_norm)

    noise_std = (config.noise_multiplier * config.clip_norm) / max(data.size(0), 1)
    for param in model.parameters():
        if param.grad is None:
            continue
        noise = torch.normal(
            mean=0.0,
            std=noise_std,
            size=param.grad.shape,
            device=param.grad.device
        )
        param.grad.add_(noise)

    return loss.item()

def train_secure(model, loader, config: DefenseConfig, epochs=2):
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    for epoch in range(epochs):
        running_loss = 0.0
        sample_count = 0
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            adv = fgsm_attack(model, data, target, epsilon=config.epsilon_adv)
            optimizer.zero_grad()
            loss_value = dp_sgd_step(model, adv, target, config)
            optimizer.step()
            running_loss += loss_value * target.size(0)
            sample_count += target.size(0)
        avg_loss = running_loss / max(sample_count, 1)
        print(f'Epoch {epoch+1}/{epochs} - secure loss: {avg_loss:.4f}')

secure_model = CNN().to(device)
secure_loader = DataLoader(sanitized_dataset, batch_size=64, shuffle=True)
config = DefenseConfig()
print(f'Defense config: {config}')
train_secure(secure_model, secure_loader, config, epochs=2)
secure_train_acc = evaluate(secure_model, train_loader)
secure_test_acc = evaluate(secure_model, test_loader)
print(f'\nSecure Model Accuracy: Train={secure_train_acc:.2f}%, Test={secure_test_acc:.2f}%')

## Secure Inference: Input Validation + Rate Limiting

- Define simple input validation constraints for inference safety.
- Implement a basic rate limiter and simulate burst requests.
- Report how many requests are allowed under the configured burst.

Expected output: a single burst-control statistic (allowed vs attempted).

In [ ]:
@dataclass
class InputPolicy:
    max_abs_value: float = 3.0
    max_variance: float = 2.0

def validate_input(x: torch.Tensor, policy: InputPolicy) -> bool:
    """Check if input passes validation policy."""
    if not torch.isfinite(x).all():
        return False
    if x.abs().max().item() > policy.max_abs_value:
        return False
    if x.var().item() > policy.max_variance:
        return False
    return True

@dataclass
class RateLimiter:
    max_qps: int = 100
    burst: int = 10
    tokens: int = 10

    def allow(self):
        if self.tokens > 0:
            self.tokens -= 1
            return True
        return False

policy = InputPolicy()
limiter = RateLimiter(tokens=10)

benign_sample, _ = test_data[0]
sponge_candidate = torch.randn_like(benign_sample) * 5.0
print(f'Benign sample accepted: {validate_input(benign_sample, policy)}')
print(f'Sponge-like sample accepted: {validate_input(sponge_candidate, policy)}')

allowed = 0
attempted = 20
for _ in range(attempted):
    if limiter.allow():
        allowed += 1

print(f'Requests allowed (burst={limiter.burst}): {allowed}/{attempted}')

## Model Inversion Attack

- Perform gradient-based model inversion for multiple classes.
- Visualize reconstructed samples and inspect confidence scores.
- Discuss what this implies for confidentiality risk.

Expected output: class-wise inversion confidence and reconstruction plots.

In [ ]:
def model_inversion_attack(model, target_class, iterations=80, learning_rate=0.1):
    """Reconstruct a representative input for target class via gradient ascent."""
    model.eval()
    x = torch.randn(1, 1, 28, 28, device=device, requires_grad=True)
    optimizer = optim.Adam([x], lr=learning_rate)

    for _ in range(iterations):
        optimizer.zero_grad()
        output = model(x)
        regularizer = 1e-3 * x.pow(2).mean()
        loss = -output[0, target_class] + regularizer
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            x.clamp_(-3, 3)

    return x.detach()

inverted_samples = {}
for target_class in range(3):
    inverted_x = model_inversion_attack(baseline_model, target_class, iterations=60)
    inverted_samples[target_class] = inverted_x
    out = baseline_model(inverted_x)
    pred_conf = torch.softmax(out, dim=1)[0, target_class].item()
    print(f'Inverted class {target_class}: confidence={pred_conf:.3f}')

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for i in range(3):
    axes[i].imshow(inverted_samples[i].squeeze().cpu().numpy(), cmap='gray')
    axes[i].set_title(f'Reconstructed Class {i}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## Synthetic Data Generation

- Train a lightweight VAE and generate synthetic MNIST-like samples.
- Visualize the generated images and comment on quality/diversity.
- Relate synthetic data to privacy/utility trade-offs.

Expected output: training logs and a grid of synthetic samples.

In [ ]:
class SimpleVAE(nn.Module):
    def __init__(self, latent_dim=10):
        super().__init__()
        self.latent_dim = latent_dim
        self.enc1 = nn.Linear(28 * 28, 256)
        self.enc_mu = nn.Linear(256, latent_dim)
        self.enc_logvar = nn.Linear(256, latent_dim)
        self.dec1 = nn.Linear(latent_dim, 256)
        self.dec_out = nn.Linear(256, 28 * 28)
    
    def encode(self, x):
        x = x.view(x.size(0), -1)
        hidden = torch.relu(self.enc1(x))
        mu = self.enc_mu(hidden)
        logvar = self.enc_logvar(hidden)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps
    
    def decode(self, z):
        hidden = torch.relu(self.dec1(z))
        return torch.sigmoid(self.dec_out(hidden))
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

def vae_loss(recon_x, x, mu, logvar):
    """VAE loss on de-normalized pixels so BCE stays valid."""
    flat_x = x.view(x.size(0), -1)
    flat_x = torch.clamp(flat_x * 0.3081 + 0.1307, 0.0, 1.0)
    bce = nn.functional.binary_cross_entropy(recon_x, flat_x, reduction='sum')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return bce + kld

vae = SimpleVAE(latent_dim=10).to(device)
vae_optimizer = optim.Adam(vae.parameters(), lr=1e-3)
vae_subset = Subset(train_dataset, train_indices[:1000])
vae_loader = DataLoader(vae_subset, batch_size=64, shuffle=True)

for epoch in range(2):
    vae.train()
    running_loss = 0.0
    sample_count = 0
    for data, _ in vae_loader:
        data = data.to(device)
        recon, mu, logvar = vae(data)
        loss = vae_loss(recon, data, mu, logvar)
        vae_optimizer.zero_grad()
        loss.backward()
        vae_optimizer.step()
        running_loss += loss.item()
        sample_count += data.size(0)
    print(f'VAE Epoch {epoch+1}/2 - loss per sample: {running_loss / max(sample_count, 1):.2f}')

with torch.no_grad():
    z = torch.randn(10, 10, device=device)
    synthetic_samples = vae.decode(z)
    synthetic_samples = synthetic_samples.view(-1, 1, 28, 28)

print(f'\nGenerated {len(synthetic_samples)} synthetic samples')

fig, axes = plt.subplots(2, 5, figsize=(12, 4))
for i in range(10):
    axes[i // 5, i % 5].imshow(synthetic_samples[i].squeeze().cpu().numpy(), cmap='gray')
    axes[i // 5, i % 5].axis('off')
plt.suptitle('Synthetic MNIST Samples (VAE-generated)')
plt.tight_layout()
plt.show()

## Attack Suite

- Evaluate multiple attacks (FGSM, PGD, MIA, sponge) on defended and baseline models.
- Add at least one stronger or complementary attack condition.
- Summarize trade-offs between clean accuracy and robustness.

Expected output: side-by-side metrics for baseline vs secure models.

In [ ]:
def get_confidences(model, loader):
    model.eval()
    confs = []
    with torch.no_grad():
        for data, _ in loader:
            data = data.to(device)
            out = model(data)
            probs = torch.softmax(out, dim=1)
            confs.extend(probs.max(dim=1)[0].cpu().numpy())
    return np.array(confs)

def eval_fgsm(model, loader, epsilon=0.1):
    model.eval()
    correct = 0
    total = 0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        adv = fgsm_attack(model, data, target, epsilon=epsilon)
        out = model(adv)
        pred = out.argmax(dim=1)
        correct += (pred == target).sum().item()
        total += target.size(0)
    return 100.0 * correct / total

def pgd_attack(model, data, target, epsilon=0.1, alpha=0.01, steps=10):
    """Iterative PGD attack with projection back into the epsilon-ball."""
    model.eval()
    delta = torch.zeros_like(data, requires_grad=True)

    for _ in range(steps):
        adv = torch.clamp(data + delta, -3, 3)
        loss = nn.CrossEntropyLoss()(model(adv), target)
        gradient = torch.autograd.grad(loss, delta)[0]
        with torch.no_grad():
            delta += alpha * gradient.sign()
            delta.clamp_(-epsilon, epsilon)
            delta.copy_(torch.clamp(data + delta, -3, 3) - data)

    return torch.clamp(data + delta.detach(), -3, 3)

def eval_pgd(model, loader, epsilon=0.1, alpha=0.01, steps=10):
    model.eval()
    correct = 0
    total = 0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        adv = pgd_attack(model, data, target, epsilon=epsilon, alpha=alpha, steps=steps)
        out = model(adv)
        pred = out.argmax(dim=1)
        correct += (pred == target).sum().item()
        total += target.size(0)
    return 100.0 * correct / total

def membership_auc(model, member_loader=train_loader, nonmember_loader=test_loader):
    train_conf = get_confidences(model, member_loader)
    test_conf = get_confidences(model, nonmember_loader)
    labels = np.concatenate([np.ones(len(train_conf)), np.zeros(len(test_conf))])
    scores = np.concatenate([train_conf, test_conf])
    return binary_roc_auc(labels, scores)

def sponge_input(scale=5.0):
    return torch.randn(1, 1, 28, 28) * scale

noise_patch = torch.tensor(
    np.random.default_rng(42).uniform(2.0, 3.0, size=(5, 5)),
    dtype=torch.float32
).unsqueeze(0)

def apply_trigger(x, pattern='corner_patch'):
    x_triggered = x.clone()
    if pattern == 'corner_patch':
        x_triggered[:, :5, :5] = 3.0
    elif pattern == 'center_pixel':
        x_triggered[:, 13:15, 13:15] = 3.0
    elif pattern == 'noise_pattern':
        x_triggered[:, -5:, -5:] = noise_patch.to(x_triggered.device)
    else:
        raise ValueError(f'Unknown trigger pattern: {pattern}')
    return x_triggered

def create_triggered_dataset(dataset, backdoor_rate=0.1, target_label=0, pattern='corner_patch'):
    indices = set(np.random.choice(len(dataset), int(backdoor_rate * len(dataset)), replace=False).tolist())
    poisoned = []
    for i in range(len(dataset)):
        x, y = dataset[i]
        if i in indices:
            poisoned.append((apply_trigger(x, pattern), target_label))
        else:
            poisoned.append((x, y))
    return poisoned, indices

def backdoor_attack_success_rate(model, dataset, target_label=0, pattern='corner_patch', sample_limit=160):
    model.eval()
    triggered_samples = []
    for i in range(len(dataset)):
        x, y = dataset[i]
        if y != target_label:
            triggered_samples.append((apply_trigger(x, pattern), target_label))
        if len(triggered_samples) >= sample_limit:
            break
    trigger_loader = DataLoader(ListDataset(triggered_samples), batch_size=64, shuffle=False)
    success = 0
    total = 0
    with torch.no_grad():
        for data, target in trigger_loader:
            data, target = data.to(device), target.to(device)
            pred = model(data).argmax(dim=1)
            success += (pred == target).sum().item()
            total += target.size(0)
    return 100.0 * success / max(total, 1)

def fit_standard_silent(model, dataset, epochs=1):
    loader = DataLoader(dataset if isinstance(dataset, Dataset) else ListDataset(dataset), batch_size=64, shuffle=True)
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            loss = criterion(model(data), target)
            loss.backward()
            optimizer.step()
    return model

def fit_secure_variant(model, dataset, config, epochs=1, adversarial=True, dp=True):
    base_dataset = dataset if isinstance(dataset, Dataset) else ListDataset(dataset)
    loader = DataLoader(base_dataset, batch_size=64, shuffle=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    for _ in range(epochs):
        model.train()
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            working_data = fgsm_attack(model, data, target, epsilon=config.epsilon_adv) if adversarial else data
            optimizer.zero_grad()
            if dp:
                dp_sgd_step(model, working_data, target, config)
            else:
                loss = criterion(model(working_data), target)
                loss.backward()
            optimizer.step()
    return model

# Small evaluation subsets keep this analysis usable on limited hardware.
small_train_loader = DataLoader(Subset(train_dataset, train_indices[:256]), batch_size=64, shuffle=False)
small_test_loader = DataLoader(Subset(test_dataset, test_indices[:256]), batch_size=64, shuffle=False)
sanitized_small = ListDataset(sanitized_data[:800])

comparison_rows = []
for model_name, model in [('Baseline', baseline_model), ('Secure', secure_model)]:
    comparison_rows.append({
        'model': model_name,
        'clean_acc': round(evaluate(model, small_test_loader), 2),
        'fgsm_0.10': round(eval_fgsm(model, small_test_loader, epsilon=0.10), 2),
        'fgsm_0.20': round(eval_fgsm(model, small_test_loader, epsilon=0.20), 2),
        'pgd_10': round(eval_pgd(model, small_test_loader, epsilon=0.10, alpha=0.02, steps=10), 2),
        'pgd_20': round(eval_pgd(model, small_test_loader, epsilon=0.10, alpha=0.01, steps=20), 2),
        'mia_auc': round(membership_auc(model, small_train_loader, small_test_loader), 3)
    })

comparison_df = pd.DataFrame(comparison_rows)
print('Baseline vs Secure Model Metrics')
print(comparison_df.to_string(index=False))

# Attack scaling: stronger FGSM should lower accuracy.
epsilon_values = [0.10, 0.15, 0.20, 0.25, 0.30]
epsilon_acc = [eval_fgsm(secure_model, small_test_loader, epsilon=eps) for eps in epsilon_values]
plt.figure(figsize=(6, 4))
plt.plot(epsilon_values, epsilon_acc, marker='o')
plt.title('Secure Model Accuracy Under FGSM Attack Strength')
plt.xlabel('FGSM epsilon')
plt.ylabel('Accuracy (%)')
plt.grid(True, alpha=0.3)
plt.show()

# PGD iteration sweep: check whether extra iterations still hurt robustness.
pgd_steps_grid = [5, 10, 20, 30]
pgd_step_results = [eval_pgd(secure_model, small_test_loader, epsilon=0.10, alpha=0.01, steps=steps) for steps in pgd_steps_grid]
pgd_steps_df = pd.DataFrame({'pgd_steps': pgd_steps_grid, 'secure_acc': np.round(pgd_step_results, 2)})
print('\nPGD iteration sweep (secure model)')
print(pgd_steps_df.to_string(index=False))

# Multi-attack ensemble: count any rejected or misclassified sample as a failure.
combined_failures = 0
combined_total = 0
for data, target in small_test_loader:
    data, target = data.to(device), target.to(device)
    fgsm_adv = fgsm_attack(secure_model, data, target, epsilon=0.10)
    combined_adv = pgd_attack(secure_model, fgsm_adv, target, epsilon=0.10, alpha=0.02, steps=5)
    outputs = secure_model(combined_adv)
    predictions = outputs.argmax(dim=1)
    for i in range(combined_adv.size(0)):
        sample = combined_adv[i].detach().cpu()
        rejected = not validate_input(sample, policy)
        misclassified = predictions[i].item() != target[i].item()
        combined_failures += int(rejected or misclassified)
        combined_total += 1

print(f'\nCombined FGSM+PGD+validation failure rate: {100.0 * combined_failures / max(combined_total, 1):.2f}%')
print(f'Sponge rejection rate (policy only): {100.0 * sum(not validate_input(sponge_input().squeeze(0), policy) for _ in range(100)) / 100:.2f}%')

# Poisoning trade-off: measure how much larger budgets hurt clean performance.
poison_rates = [0.01, 0.05, 0.10, 0.20, 0.30]
poison_base = ListDataset([train_data[i] for i in range(800)])
poison_tradeoff_rows = []
for rate in poison_rates:
    label_flip_data, _ = poison_labels(poison_base, poison_rate=rate, target_label=0)
    label_flip_model = fit_standard_silent(CNN().to(device), ListDataset(label_flip_data), epochs=1)
    backdoor_data, _ = create_triggered_dataset(poison_base, backdoor_rate=rate, target_label=0, pattern='corner_patch')
    backdoor_model = fit_standard_silent(CNN().to(device), ListDataset(backdoor_data), epochs=1)
    poison_tradeoff_rows.append({
        'poison_rate': rate,
        'label_flip_acc': round(evaluate(label_flip_model, small_test_loader), 2),
        'backdoor_clean_acc': round(evaluate(backdoor_model, small_test_loader), 2),
        'backdoor_asr': round(backdoor_attack_success_rate(backdoor_model, test_data, target_label=0, pattern='corner_patch', sample_limit=160), 2)
    })

poison_tradeoff_df = pd.DataFrame(poison_tradeoff_rows)
print('\nPoisoning budget trade-off')
print(poison_tradeoff_df.to_string(index=False))
threshold_rows = poison_tradeoff_df[(poison_tradeoff_df['label_flip_acc'] < comparison_df.loc[comparison_df['model'] == 'Baseline', 'clean_acc'].iloc[0] - 5) | (poison_tradeoff_df['backdoor_asr'] > 60)]
if not threshold_rows.empty:
    first_threshold = threshold_rows.iloc[0]['poison_rate']
    print(f'Approximate poisoning threshold: {first_threshold:.2f}')
else:
    print('Approximate poisoning threshold: above 0.30 for this lightweight experiment')

# Compare different trigger shapes for stealth and effectiveness.
trigger_patterns = ['corner_patch', 'center_pixel', 'noise_pattern']
trigger_rows = []
for pattern in trigger_patterns:
    trigger_train_data, _ = create_triggered_dataset(poison_base, backdoor_rate=0.10, target_label=0, pattern=pattern)
    trigger_model = fit_standard_silent(CNN().to(device), ListDataset(trigger_train_data), epochs=1)
    trigger_rows.append({
        'pattern': pattern,
        'clean_acc': round(evaluate(trigger_model, small_test_loader), 2),
        'attack_success_rate': round(backdoor_attack_success_rate(trigger_model, test_data, target_label=0, pattern=pattern, sample_limit=160), 2)
    })

trigger_df = pd.DataFrame(trigger_rows)
print('\nTrojan trigger comparison')
print(trigger_df.to_string(index=False))
print(f"Stealthiest trigger: {trigger_df.sort_values('clean_acc', ascending=False).iloc[0]['pattern']}")
print(f"Most effective trigger: {trigger_df.sort_values('attack_success_rate', ascending=False).iloc[0]['pattern']}")

# Privacy-utility sweep: higher noise usually improves privacy and harms accuracy.
privacy_rows = []
for noise_multiplier in [0.5, 1.0, 1.5, 2.0]:
    sweep_config = DefenseConfig(epsilon_adv=0.10, clip_norm=1.0, noise_multiplier=noise_multiplier)
    privacy_model = fit_secure_variant(CNN().to(device), sanitized_small, sweep_config, epochs=1, adversarial=True, dp=True)
    privacy_rows.append({
        'noise_multiplier': noise_multiplier,
        'test_acc': round(evaluate(privacy_model, small_test_loader), 2),
        'mia_auc': round(membership_auc(privacy_model, small_train_loader, small_test_loader), 3)
    })

privacy_df = pd.DataFrame(privacy_rows)
print('\nPrivacy vs utility sweep')
print(privacy_df.to_string(index=False))
best_privacy_row = privacy_df.sort_values(['mia_auc', 'test_acc'], ascending=[True, False]).iloc[0]
print(f"Selected privacy-utility operating point: noise={best_privacy_row['noise_multiplier']}, acc={best_privacy_row['test_acc']}, mia_auc={best_privacy_row['mia_auc']}")
plt.figure(figsize=(6, 4))
plt.scatter(privacy_df['mia_auc'], privacy_df['test_acc'], s=80)
for _, row in privacy_df.iterrows():
    plt.annotate(f"noise={row['noise_multiplier']}", (row['mia_auc'], row['test_acc']), textcoords='offset points', xytext=(5, 5))
plt.xlabel('Membership inference AUC')
plt.ylabel('Test accuracy (%)')
plt.title('Privacy-Utility Trade-off')
plt.grid(True, alpha=0.3)
plt.show()

# Defense ablation: identify which component matters most for PGD robustness.
ablation_rows = []
dp_only_model = fit_secure_variant(CNN().to(device), sanitized_small, DefenseConfig(noise_multiplier=1.0), epochs=1, adversarial=False, dp=True)
ablation_rows.append({
    'defense': 'Only DP-SGD',
    'clean_acc': round(evaluate(dp_only_model, small_test_loader), 2),
    'pgd_20': round(eval_pgd(dp_only_model, small_test_loader, epsilon=0.10, alpha=0.01, steps=20), 2)
})

adv_only_model = fit_secure_variant(CNN().to(device), sanitized_small, DefenseConfig(noise_multiplier=0.0), epochs=1, adversarial=True, dp=False)
ablation_rows.append({
    'defense': 'Only adversarial training',
    'clean_acc': round(evaluate(adv_only_model, small_test_loader), 2),
    'pgd_20': round(eval_pgd(adv_only_model, small_test_loader, epsilon=0.10, alpha=0.01, steps=20), 2)
})

sanitize_only_model = fit_standard_silent(CNN().to(device), sanitized_small, epochs=1)
ablation_rows.append({
    'defense': 'Only input sanitization',
    'clean_acc': round(evaluate(sanitize_only_model, small_test_loader), 2),
    'pgd_20': round(eval_pgd(sanitize_only_model, small_test_loader, epsilon=0.10, alpha=0.01, steps=20), 2)
})

ablation_df = pd.DataFrame(ablation_rows).sort_values('pgd_20', ascending=False)
print('\nDefense ablation ranking by PGD robustness')
print(ablation_df.to_string(index=False))
print(f"Highest PGD robustness: {ablation_df.iloc[0]['defense']}")

### Attack Scaling
TODO: Increase FGSM attack strength (ε) from 0.1 to 0.3 in steps of 0.05 and plot secure model accuracy vs ε.

### PGD Iterations
TODO: Vary PGD attack steps (5, 10, 20, 50) while keeping ε=0.1.  
Report accuracy for each step count. Does accuracy continue to drop beyond 20 steps?

### Multi-Attack Ensemble
TODO: Combine FGSM + PGD + Sponge attacks simultaneously.  
What % of test samples are rejected/misclassified under combined attack?

### Poisoning Budget Trade-off
TODO: Vary poison rate (1%, 5%, 10%, 20%, 30%) on label-flip and backdoor attacks.  
For each rate, measure clean accuracy and target misclassification rate. Find the poisoning threshold.

### Trojan Triggers
TODO: Create trojans with different trigger patterns (corner patch, center pixel, noise pattern).  
Which trigger is most stealthy (maintains clean accuracy)? Which is most effective (highest ASR)?

### Privacy-Utility Trade-off
TODO: Sweep DP noise multiplier (0.5, 1.0, 1.5, 2.0) and measure:
- Test accuracy (utility)
- Membership inference AUC (privacy)
Create a scatter plot of accuracy vs MIA-AUC. What's the optimal ε_dp for your threat model?

### Defense Ablation 
TODO: Disable each defense component and measure accuracy + robustness:
- Only DP-SGD (no adversarial training)
- Only adversarial training (no DP-SGD)  
- Only input sanitization (no adversarial training)  
Rank defenses by impact on PGD robustness. Which is most important?

## Deployment Checklist: Production Security Operationalization

- Translate technical defenses into production controls.
- Define monitoring, incident response, retraining cadence, and governance checks.
- Keep checklist items measurable and auditable.

Expected output: categorized security checklist suitable for deployment review.

In [ ]:
deployment_checklist = {
    'Pre-deployment': [
        'Model audit completed with documented architecture, dependencies, and checksum verification.',
        'Privacy audit approved, including DP-SGD configuration review and membership-inference threshold check.',
        'Trojan and backdoor scan performed on training data and model activations before release.',
        'Adversarial robustness test passed for FGSM and PGD against the production acceptance threshold.',
        'Synthetic-data quality review completed with sign-off on privacy and representational fidelity.'
    ],
    'Runtime operations': [
        'Input validation blocks malformed, high-variance, or out-of-range samples before inference.',
        'Rate limiting is enabled per client and burst limits are monitored with alert thresholds.',
        'Inference logs capture request metadata, prediction confidence, and rejection reasons.',
        'Model versioning and artifact lineage are recorded for every deployed checkpoint.',
        'Latency and resource-usage monitoring are active to detect sponge-style degradation.'
    ],
    'Detection & response': [
        'Anomaly detection flags shifts in confidence, class distribution, and request patterns.',
        'Scheduled trojan-detection job reviews activation clusters and trigger-like signatures.',
        'Poisoning detection monitors label drift, provenance failures, and sudden class imbalance.',
        'Incident response playbook defines containment, rollback, forensics, and owner escalation paths.',
        'Security notifications route to engineering and compliance teams with severity-based SLAs.'
    ],
    'Governance': [
        'GDPR and healthcare-data handling controls are mapped to the deployed pipeline.',
        'Audit trail preserves training data sources, approvals, experiments, and deployment events.',
        'Model card documents intended use, attack surface, validation results, and limitations.',
        'Threat model is reviewed at each release and updated after incidents or architecture changes.',
        'Security training for operators and reviewers is tracked and refreshed on a fixed schedule.'
    ],
    'Continuous improvement': [
        'Retraining cadence is defined with drift triggers and clean-data validation gates.',
        'Defense ablation studies are rerun after major pipeline or architecture changes.',
        'Benchmark suite tracks clean accuracy, robustness, privacy, and operational resilience over time.',
        'Recurring privacy audits verify that inference leakage remains within policy bounds.',
        'Red-team exercises simulate evasion, poisoning, privacy, and denial-of-service scenarios.'
    ]
}

checklist_rows = []
for category, items in deployment_checklist.items():
    for idx, item in enumerate(items, start=1):
        checklist_rows.append({'category': category, 'item_no': idx, 'control': item})

checklist_df = pd.DataFrame(checklist_rows)
print(checklist_df.to_string(index=False))